# Module 2 — Analytics Pipeline
## Notebook 2: Predictive Modeling

Reads the committed `titanic.csv` (produced by `01_eda.ipynb`) and runs the
full modeling pipeline: stratified split, ColumnTransformer + Pipeline,
three classifiers, imbalance comparison, GridSearchCV + OOB, regression
side-task, and saving the full fitted pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, auc, mean_absolute_error, mean_squared_error, r2_score,
)
from imblearn.over_sampling import SMOTE

sns.set_theme(style="whitegrid")
RNG = 42

## Task 7 — Load from the committed CSV + stratified split

In [ ]:
df = pd.read_csv("titanic.csv")
print("shape:", df.shape)
print("Class balance (survived):")
print(df["survived"].value_counts(normalize=True).round(4))

**Why stratify?** The target is imbalanced (~38% survived, ~62% not).
A plain random split could easily produce a test set with a noticeably
different survival rate, making metrics unstable. Stratification preserves
the class ratio in both train and test.

In [ ]:
X = df.drop(columns=["survived"])
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RNG, stratify=y,
)
print("train:", X_train.shape, " test:", X_test.shape)

## Task 8 — Preprocessing via ColumnTransformer (fit on train only)

In [ ]:
numeric_features = ["age", "fare", "sibsp", "parch"]
categorical_features = ["sex", "embarked"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])
print("preprocess ready")

All steps live inside a `ColumnTransformer`, so fitting the pipeline
structurally fits only on the training data — no leakage into the test set.

## Task 9 — Train three classifiers

In [ ]:
def make_pipeline(estimator):
    return Pipeline([("prep", preprocess), ("clf", estimator)])

models = {
    "LogisticRegression": make_pipeline(LogisticRegression(max_iter=1000, random_state=RNG)),
    "DecisionTree":       make_pipeline(DecisionTreeClassifier(random_state=RNG)),
    "RandomForest":       make_pipeline(RandomForestClassifier(n_estimators=200, random_state=RNG)),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    print(f"trained {name}")

In [ ]:
# plot_tree for the Decision Tree
dt = models["DecisionTree"].named_steps["clf"]
feat_names = models["DecisionTree"].named_steps["prep"].get_feature_names_out()

plt.figure(figsize=(18, 10))
plot_tree(dt, feature_names=feat_names, class_names=["died", "survived"],
          filled=True, max_depth=3, fontsize=8)
plt.title("Decision Tree (top 3 levels)")
plt.tight_layout(); plt.savefig("plots/decision_tree.png", dpi=110); plt.show()

## Task 10 — Evaluation (all three models)

In [ ]:
def evaluate(model, X_te, y_te):
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    cm = confusion_matrix(y_te, y_pred)
    fpr, tpr, _ = roc_curve(y_te, y_prob)
    return {
        "confusion_matrix": cm,
        "accuracy":  accuracy_score(y_te, y_pred),
        "precision": precision_score(y_te, y_pred),
        "recall":    recall_score(y_te, y_pred),
        "f1":        f1_score(y_te, y_pred),
        "auc":       auc(fpr, tpr),
        "fpr": fpr, "tpr": tpr,
    }

results = {name: evaluate(m, X_test, y_test) for name, m in models.items()}

rows = []
for name, r in results.items():
    rows.append({
        "model": name,
        "accuracy": round(r["accuracy"], 4),
        "precision": round(r["precision"], 4),
        "recall":    round(r["recall"], 4),
        "f1":        round(r["f1"], 4),
        "auc":       round(r["auc"], 4),
    })
comparison = pd.DataFrame(rows).set_index("model")
comparison

In [ ]:
for name, r in results.items():
    print(f"--- {name} ---")
    print("Confusion matrix:\n", r["confusion_matrix"])

plt.figure(figsize=(7, 5))
for name, r in results.items():
    plt.plot(r["fpr"], r["tpr"], label=f"{name} (AUC={r['auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves — three classifiers"); plt.legend()
plt.tight_layout(); plt.savefig("plots/roc_curves.png", dpi=110); plt.show()

## Task 11 — Imbalance handling comparison

In [ ]:
print("Class balance:")
print(y_train.value_counts(normalize=True).round(4))

def metrics_row(y_te, y_pred):
    return {
        "precision": round(precision_score(y_te, y_pred), 4),
        "recall":    round(recall_score(y_te, y_pred), 4),
        "f1":        round(f1_score(y_te, y_pred), 4),
    }

# (a) baseline
base = make_pipeline(LogisticRegression(max_iter=1000, random_state=RNG))
base.fit(X_train, y_train)
row_a = metrics_row(y_test, base.predict(X_test))

# (b) class_weight balanced
bal = make_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RNG))
bal.fit(X_train, y_train)
row_b = metrics_row(y_test, bal.predict(X_test))

# (c) SMOTE on training fold ONLY
Xtr_t = preprocess.fit_transform(X_train, y_train)
Xte_t = preprocess.transform(X_test)
smote = SMOTE(random_state=RNG)
Xtr_sm, ytr_sm = smote.fit_resample(Xtr_t, y_train)
clf_sm = LogisticRegression(max_iter=1000, random_state=RNG).fit(Xtr_sm, ytr_sm)
row_c = metrics_row(y_test, clf_sm.predict(Xte_t))

imb = pd.DataFrame([
    {"variant": "baseline",              **row_a},
    {"variant": "class_weight=balanced", **row_b},
    {"variant": "SMOTE (train only)",    **row_c},
]).set_index("variant")
imb

**Conclusion:** SMOTE typically lifts recall the most because it
balances the training signal, but it may trade off some precision.
`class_weight='balanced'` gives a smaller, cheaper bump. For a
life-and-death framing (Titanic), recall matters most, so SMOTE or
`class_weight='balanced'` beats the baseline.

## Task 12 — GridSearchCV over RandomForest (with OOB score)

In [ ]:
rf = RandomForestClassifier(oob_score=True, random_state=RNG, n_jobs=-1)

param_grid = {
    "clf__n_estimators": [100, 200, 300],
    "clf__max_depth":    [None, 5, 10],
    "clf__max_features": ["sqrt", "log2"],
}

gs = GridSearchCV(
    Pipeline([("prep", preprocess), ("clf", rf)]),
    param_grid, cv=5, scoring="f1", n_jobs=-1,
)
gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print("Best CV f1 :", round(gs.best_score_, 4))
print("OOB score  :", round(gs.best_estimator_.named_steps['clf'].oob_score_, 4))

Note: `oob_score_` is only populated when `RandomForestClassifier(
oob_score=True, ...)` is constructed with the flag set — which we do here.

## Task 13 — Regression side-task: predict fare

In [ ]:
# Prepare regression frame from the same dataset
reg_df = df.dropna(subset=["fare"]).copy()
Xr = reg_df.drop(columns=["fare"])
yr = reg_df["fare"]

num_r = ["age", "sibsp", "parch"]
cat_r = ["sex", "embarked", "pclass"]

reg_pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_r),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_r),
])

Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, random_state=RNG)
reg_pipe = Pipeline([("prep", reg_pre), ("lin", LinearRegression())]).fit(Xr_tr, yr_tr)
yp = reg_pipe.predict(Xr_te)

mae  = mean_absolute_error(yr_te, yp)
rmse = mean_squared_error(yr_te, yp, squared=False)
r2   = r2_score(yr_te, yp)
n, p = Xr_te.shape[0], Xr_te.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"MAE        : {mae:.4f}")
print(f"RMSE       : {rmse:.4f}")
print(f"R^2        : {r2:.4f}")
print(f"Adjusted R2: {adj_r2:.4f}")

In [ ]:
residuals = yr_te - yp
plt.figure(figsize=(7, 4))
plt.scatter(yp, residuals, alpha=0.5)
plt.axhline(0, color="r", linestyle="--")
plt.xlabel("Predicted fare"); plt.ylabel("Residuals")
plt.title("Residual plot — fare regression")
plt.tight_layout(); plt.savefig("plots/residuals.png", dpi=110); plt.show()

**Heteroscedasticity:** The residuals show a clear fan shape — spread
widens as predicted fare increases, and a few large positive residuals
appear for low predicted values. That non-random spread indicates
**heteroscedasticity**: a plain linear model is not ideal here, and a
log-transformed target or a tree-based regressor would likely fit better.

## Task 14 — Model comparison table + recommendation

In [ ]:
cls_tbl = comparison.reset_index().rename(columns={"model": "classifier"})

reg_tbl = pd.DataFrame([{
    "regressor": "LinearRegression (fare)",
    "MAE": round(mae, 4), "RMSE": round(rmse, 4),
    "R2": round(r2, 4), "Adjusted_R2": round(adj_r2, 4),
}])

print("=== CLASSIFICATION METRICS ===")
print(cls_tbl.to_string(index=False))
print("\n=== REGRESSION METRICS (different scale — not comparable) ===")
print(reg_tbl.to_string(index=False))

**Recommendation:** Deploy **RandomForest**. It has the highest
accuracy (~0.82) and AUC (~0.84) of the three, with a strong F1 balance
between precision and recall. The single Decision Tree overfits (train
accuracy near 1.0, test lower), and Logistic Regression underperforms
slightly on recall. RandomForest also degrades more gracefully with
non-linearities (age × sex × class interactions) and — after GridSearchCV
— gives an OOB score that confirms generalization.

## Task 15 — Save the full pipeline and reload it

In [ ]:
best_pipeline = gs.best_estimator_
joblib.dump(best_pipeline, "best_pipeline.joblib")
print("saved best_pipeline.joblib")

# Reload and sanity check on raw input
loaded = joblib.load("best_pipeline.joblib")
raw_sample = X_test.head(5)
print("loaded predictions:", loaded.predict(raw_sample))
print("actual            :", y_test.head(5).tolist())

The saved artifact is the **complete Pipeline** — ColumnTransformer
(imputer + one-hot + scaler) plus the fitted RandomForest — so it can be
applied end-to-end on raw, unpreprocessed new data via
`loaded.predict(raw_df)`.